In [0]:
import os
import requests
import z_pipelineUtils as z_pU

# File that stores the most recent run details of the pipeline
PIPELINE_STATE = "/Volumes/nc_pipeline/default/pipeline_data/pipeline_state.json"
# Directory that will store the downloaded zip files
DOWNLOAD_DEST = "/Volumes/nc_pipeline/default/pipeline_data/zipped"

saved_state, live_state = z_pU.readStateJson(PIPELINE_STATE)

for voter_file in saved_state:
    url = saved_state[voter_file]["url"]
    content_length = int(saved_state[voter_file]["content_length"])
    downloaded = saved_state[voter_file]["downloaded"]

    if downloaded is False:
        try:
            with requests.get(url, stream=True, timeout=60) as r:
                r.raise_for_status()
                full_zip_path = os.path.join(DOWNLOAD_DEST, voter_file)
                with open(full_zip_path, "wb") as file:
                    # I'm not sure what the optimal chunk size is (probably dynamic from looking at available RAM?)
                    # I'm dividing the full size by 10 so it can, hopefully, report progress in increments of 10
                    # There may be more prints than 10, chunk_size sets the max size a chunk can be
                    # The network or server may  provide fewer chunks on any given iteration
                    chunk_size = content_length // 10
                    progress = 0
                    print(f"Beginning {voter_file} download.")
                    print("Download progress: 0%")
                    for chunk in r.iter_content(chunk_size=chunk_size):
                        file.write(chunk)
                        progress += len(chunk)
                        percent = (progress / content_length) * 100
                        print(f"Download progress: {percent:.0f}%")
                    print("Download complete.")
                    live_state[voter_file]["downloaded"] = True
                    z_pU.updateStateJson(live_state, PIPELINE_STATE)
        except (requests.exceptions.RequestException, IOError) as e:
            print(f"Download failed: {e}")
